In [4]:
from utils import make_nanoribbon, hamiltonian
import numpy as np
import matplotlib.pyplot as plt
from ase.visualize import view
import sisl
from sisl.physics import RecursiveSI
import pandas as pd
from numba import njit
from scipy.sparse import csc_array

%load_ext line_profiler


# Make electrode

In [35]:
WIDTH_ELEC = 6
LENGTH_ELEC = 3
electrode = make_nanoribbon(WIDTH_ELEC, LENGTH_ELEC)
electrode.plot(axes="xy")

# Make device

In [58]:
LENGTH_DEVICE = 1 # Number of 'electrode' sizes to include **between** electrodes
device = electrode.tile(2+LENGTH_DEVICE, 0)
N = len(electrode)
atoms_idx = list(range(len(device)))
device.plot(axes="xy", atoms_style={"atoms": atoms_idx[:N] + atoms_idx[-N:] , "color": "red"})

In [132]:

L, R = SE.self_energy_lr(E=0)

def pretty_print_columns(A, decimals=2, zero_repr="0"):
    """
    Print a 2D NumPy array with per-column alignment.
    Handles complex numbers, negatives, and zeros gracefully.
    """
    A = np.asarray(A)
    if A.ndim != 2:
        raise ValueError("Input must be a 2D array")

    rows, cols = A.shape
    is_complex = np.iscomplexobj(A)

    # Build a string matrix (formatted values)
    str_matrix = np.empty(A.shape, dtype=object)
    for i in range(rows):
        for j in range(cols):
            val = A[i, j]
            if val == 0:
                s = zero_repr
            elif is_complex:
                s = f"{val.real:.{decimals}f}{val.imag:+.{decimals}f}j"
            else:
                s = f"{val:.{decimals}f}"
            str_matrix[i, j] = s

    # Compute per-column widths
    col_widths = [max(len(str_matrix[i, j]) for i in range(rows)) for j in range(cols)]

    # Print rows with proper per-column alignment
    for i in range(rows):
        row_str = " ".join(str_matrix[i, j].rjust(col_widths[j]) for j in range(cols))
        print(row_str)

        
print("H (no PBC)")
pretty_print_columns(Ham.Hk(format="array"), decimals=2)

H (no PBC)
    0     0     0 -2.70     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
    0     0     0 -2.70 -2.70     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0

In [133]:
print("L")
pretty_print_columns(L, decimals=2)

L
0.00-31752.30j 0.00+25463.36j 0.00-14131.10j 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
0.00+25463.36j 0.00-20420.03j 0.00+11332.26j 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
0.00-14131.10j 0.00+11332.26j  0.00-6288.93j 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
             0              0              0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
             0              0              0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
             0              0              0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
             0              0              0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0          0          0          0
             0              0   

In [134]:
print("R")
pretty_print_columns(R, decimals=1)

R
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0             0             0             0
0 0 0 0 0 0 0 0 0 0 0 0 0 

In [135]:
# Nk = 15
# E_list = np.arange(-1, 1, 0.1)
# eta = 1e-5

# SEL = sisl.RecursiveSI(PBC_ham, infinite="-A", eta=eta) #H0
# SER = sisl.RecursiveSI(PBC_ham, infinite="+A", eta=eta)

# T_k_sum = np.zeros([len(E_list), Nk], dtype=float) # store T(E,k)
# kpts = sisl.MonkhorstPack(Ham, [1, Nk, 1]).k
# print("kpts:\n", kpts)

# for ik, kvec in enumerate(kpts):
#     # print(ik)
#     # get k-dependent H and S as arrays
#     Hk = Ham.Hk(k=kvec, format="array")
#     Sk = Ham.Sk(k=kvec, format="array")
    
#     for ie, en in enumerate(E_list):
#         # print(ie)
#         En = en + 1j*eta
#         # compute surface self-energies for this k:
        
#         SL, SR = SE.self_energy_lr(E=En, k=kvec)
#         # SL, SR = SEL.self_energy(E=En, k=kvec), SER.self_energy(E=En, k=kvec)
        
#         # compute gammas 
#         GammaL_k = 1j*(SL - SL.conj())
#         GammaR_k = 1j*(SR - SR.conj())
        
#         # atom indicies
#         nL = len(GammaL_k)
#         iL = slice(0, nL)
#         nR = len(GammaR_k)
#         iR = slice(0, nR)
        
#         invG = Sk*En - Hk
        
#         invG[iL, iL] -= SL
#         invG[iR, iR] -= SR
        
#         G = np.linalg.solve(invG, np.eye(invG.shape[0]))
#         # break
#         AL = G[:, iL] @ GammaL_k @ G[:, iL].T.conj()
#         AR = G[:, iR] @ GammaR_k @ G[:, iR].T.conj()
        
        
#         T_k = np.trace(GammaR_k @ AR).real
#         T_k_sum[ie, ik] = T_k
        
#         if ie == 0 and ik == 0:
#             print(f"is AL and AR identical : {np.all(AL == AR)}")
#             print(f"{nL = }, {nR = }")
#             continue
#             print(f"{GammaL_k.shape = }")
#             print(f"{GammaR_k.shape = }")
#             print(f"{G.shape = }")
#             print(f"{SL.shape = }")
#             print(f"{SR.shape = }")
#             print(f"{invG[iR, iR].shape = }")


# pretty_print_columns(T_k_sum, decimals=3) # no k-dependece in transport (expected for periodic systems / no defects)

In [ ]:
H_0 = hamiltonian(electrode)
H_D = hamiltonian(device)
H_D.set_nsc([1, 1, 1]) # device is not periodic since we use self-energies for edges/electrode
print(f"{H_0.nsc = },\t expected : [3,1,1]")
print(f"{H_D.nsc = },\t expected : [1,1,1]")

H_0.nsc = array([3, 1, 1], dtype=int32),	 expected : [3,1,1]
H_D.nsc = array([1, 1, 1], dtype=int32),	 expected : [1,1,1]


In [91]:
eta = 1e-5
SE = RecursiveSI(H_0, infinite="+A", eta=eta)
s = SE.self_energy(E=0)
GL = SE.se2broadening(L)

In [ ]:
def _direction(**kwargs):
    Nk = kwargs.get("Nk", 1)
    axis = kwargs.get("axis", 1)
    if isinstance(axis, int):
        if not axis in range(3): # 0, 1, or 2
            raise ValueError("'axis' must be  0,  1,  or  2.")
        d = [1, 1, 1]
        d[axis] = Nk
        return d, Nk
    else:
        raise ValueError("axis must be  int.")
        
@njit
def hermconj(matrix):
    assert matrix.ndim == 2, "matrix must be 2D"
    return matrix.T.conj()

@njit
def calc_gamma(se):
    "Compute left/right gamma from left/right self-energy"
    return 1j*(se - hermconj(se))


@njit
def greens(left, right, E, H):
    GammaL = calc_gamma(left)
    GammaR = calc_gamma(right)
    N = len(GammaL)
    H[ :N,  :N] -= GammaL
    H[-N:, -N:] -= GammaR
    
    invG = E - H
    return np.linalg.inv(invG), GammaL, GammaR
    I = np.eye(invG.shape[0], dtype=invG.dtype)
    return np.linalg.solve(invG, I), GammaL, GammaR

@njit
def spectral(Greens, Gamma):
    return Greens @ Gamma @ hermconj(Greens)

@njit
def _T(AR, GammaL):
    return np.trace(AR @ GammaL)

def transport(device, electrode, energies, **kwargs):
    eta = kwargs.get("eta", 1e-5)
    k_direction, Nk = _direction(**kwargs)
    
    
    kpts = sisl.MonkhorstPack(device, k_direction).k
    NE = len(energies)
    T_k_sum = np.zeros(shape=(Nk, NE), dtype=float)
    A_ek = np.empty(shape=(Nk, NE, *electrode.Hk(format="array").shape), dtype=complex)
    
    if isinstance(energies, (float, int)):
        energies = list(energies)
    
    SE = RecursiveSI(electrode, "+A", eta=eta)
    
    for iter_k, kvec in enumerate(kpts):
        Hk = device.Hk(k=kvec, format="array")
        Sk = device.Sk(k=kvec, format="array")
        
        for iter_E, E in enumerate(energies):
            En = E + 1j*eta
            SE_L, SE_R = SE.self_energy_lr(E=En)
            
            
            G, Gamma_Lk, Gamma_Rk = greens(left=SE_L, right=SE_R, E=En*Sk, H=Hk)
            AL = spectral(G, Gamma_Lk)
            AR = spectral(G, Gamma_Rk)
            
            A_ek[iter_k, iter_E, ...] = AL + AR
            
            T_k_sum[iter_k, iter_E] = _T(AR, Gamma_Lk).real
    return T_k_sum, A_ek, kpts


H_0.nsc = array([3, 1, 1], dtype=int32),	 expected : [3,1,1]
H_D.nsc = array([1, 1, 1], dtype=int32),	 expected : [1,1,1]


In [ ]:
def LDOS(H, energies, **kwargs):
    T, A_ek, kpts = transport(H, energies, **kwargs)
    print(f"{A_ek.shape = }")
    assert A_ek.shape[1] == len(energies), "number of energies and corresponding dimension of A does not match (check implementation)"
    
    def rho(kidx, Eidx):
        """Find LDOS from diagonal of spectral function/matrix"""
        return np.diag(A_ek[kidx, Eidx]).real / (2*np.pi)
    
    LDOS = np.zeros(shape=(*A_ek.shape[:2], A_ek.shape[-1]), dtype=float) # number of (k, E, A.shape) 
    for iter_E, E in zip(range(A_ek.shape[1]), energies):
        for iter_k in range(A_ek.shape[0]):
            LDOS[iter_k, iter_E, :] = rho(iter_k, iter_E)
    
    return T, LDOS, kpts

In [120]:
Nk = 1
NE = 50
energies = np.linspace(-1, 1, num=NE)
T, ldos, kpts = LDOS(PBC_ham, energies); print(ldos.shape)

A_ek.shape = (1, 50, 96, 96)
(1, 50, 96)


In [ ]:
import ipywidgets
from ipywidgets import interact
from fractions import Fraction

def plotLDOS(ldos, kidx, site):
    if isinstance(kidx, ipywidgets.Dropdown):
        kidx = kidx.value
    fig, ax = plt.subplots(1,1)
    
    ymin, ymax = np.min(ldos), np.max(ldos)
    
    ax.plot(energies, ldos[kidx, :, site])
    ax.set_ylim(-ymax*0.1, ymax*1.01)
    ax.set_xlabel("E")
    ax.set_ylabel("LDOS")
    
    # ax.legend()
options = [(f"{[str(Fraction(val).limit_denominator(len(kpts)*2)) for val in kvec]})", i) for i, kvec in enumerate(kpts)]
ks = ipywidgets.Dropdown(options=options, description="k")
static = ipywidgets.fixed(ldos)
sites = ipywidgets.IntSlider(min=0, max=ldos.shape[-1]-1, value=0, description="Site")

interact(plotLDOS, ldos=static, kidx=ks, site=sites)
    

interactive(children=(Dropdown(description='k', options=(("['0', '0', '0'])", 0),), value=0), IntSlider(value=…

<function __main__.plotLDOS(ldos, kidx, site)>

### Profiling

In [52]:
def inverse1(G):
    return np.linalg.inv(G)
def inverse2(G):
    I = np.eye(G.shape[0])
    return np.linalg.solve(G, I)
def func(G):
    inverse1(G)
    inverse2(G)    
    # return np.linalg.solve(G, I)
%lprun -f func func(G)

Timer unit: 1e-09 s

Total time: 0.0629423 s
File: /tmp/ipykernel_282758/1137498103.py
Function: func at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def func(G):
     7         1   11338174.0 1.13e+07     18.0      inverse1(G)
     8         1   51604130.0 5.16e+07     82.0      inverse2(G)    
     9                                               # return np.linalg.solve(G, I)

In [61]:
%lprun -f transport -s -u 1e-3 LDOS(PBC_ham, energies)

A_ek.shape = (1, 11, 300, 300)


Timer unit: 0.001 s

Total time: 29.6675 s
File: /tmp/ipykernel_282758/1335168255.py
Function: transport at line 41

Line #      Hits         Time  Per Hit   % Time  Line Contents
    41                                           def transport(H, energies, **kwargs):
    42         1          0.0      0.0      0.0      eta = kwargs.get("eta", 1e-5)
    43         1          0.0      0.0      0.0      k_direction, Nk = _direction(**kwargs)
    44                                           
    45                                           
    46         1          0.8      0.8      0.0      kpts = sisl.MonkhorstPack(H, k_direction).k
    47         1          0.0      0.0      0.0      NE = len(energies)
    48         1          0.0      0.0      0.0      T_k_sum = np.zeros(shape=(Nk, NE), dtype=float)
    49         1          0.3      0.3      0.0      A_ek = np.empty(shape=(Nk, NE, *H.Hk(format="array").shape), dtype=complex)
    50                                           
    51   